<a href="https://colab.research.google.com/github/romoliangelica314/eye2ai/blob/main/eye2ai_MLProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h> Importing necessary libraries: </h>

In [ ]:
import pandas as pd
import numpy as np
import itertools
import warnings

import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

from scipy.stats import randint, uniform, loguniform

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (FunctionTransformer, RobustScaler,
StandardScaler, OneHotEncoder, OrdinalEncoder)
from sklearn.model_selection import (train_test_split, KFold,
RepeatedKFold, RandomizedSearchCV, cross_validate, learning_curve,
validation_curve, cross_val_score, cross_val_predict)
#why not stratifiedkfold?

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
root_mean_squared_error, r2_score, make_scorer)

warnings.filterwarnings("ignore")

## Dataset overview:

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/bodycam_interactions_dataset.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/bodycam_interactions_dataset.csv'

In [ ]:
print("Shape:", df.shape)
display(df.head())

In [ ]:
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
print("\nDtypes:")
display(df.dtypes)

In [ ]:
missing_markers = ["", " ", "NA", "N/A", "na", "null", "None", "unknown", "Unknown"]

In [ ]:
df = df.replace(missing_markers, np.nan)

In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({"missing_count": missing_counts,
                                "missing_pct": missing_pct})

display(missing_summary[missing_summary["missing_count"] > 0])

In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["interaction_score"], bins=30, kde=True)
plt.title("Distribution of Interaction Score")
plt.xlabel("interaction_score")
plt.ylabel("Count")
plt.show()
plt.figure(figsize=(8, 2.5))
sns.boxplot(x=df["interaction_score"])
plt.title("Boxplot of Interaction Score")
plt.show()

In [ ]:
df["interaction_score"].describe()

In [ ]:
msno.matrix(df)
plt.show()

In [ ]:
print(df.info())

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

In [ ]:
for col in cat_cols:
    print(f"{col}")
    display(df[col].value_counts(dropna=False).head(15))

In [ ]:
df[num_cols].hist(figsize=(16, 12), bins=25)
plt.tight_layout()
plt.show()

In [ ]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

In [ ]:
for col in cat_cols:
    print(f"\n{col}:")
    print(df[col].dropna().unique()[:20])

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean["suspect_age"] = pd.to_numeric(df_clean["suspect_age"], errors="coerce")
print(df_clean["suspect_age"].dtype)
display(df_clean["suspect_age"].describe())

In [ ]:
df_clean["suspect_gender"] = (df_clean["suspect_gender"].astype(str)
.str.strip().str.lower().replace({"m": "male",
                                  "f": "female"}))
display(df_clean["suspect_gender"].value_counts(dropna=False))

In [ ]:
df_clean["time_of_day"] = (
    df_clean["time_of_day"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "late night": "late_night",
        "latenight": "late_night"
    })
)

display(df_clean["time_of_day"].value_counts(dropna=False))

In [ ]:
df_clean["suspected_offense_type"] = (
    df_clean["suspected_offense_type"].astype(str).str.strip()
    .str.lower().replace({
        "mental health crisis": "mental_health_crisis",
        "behavioral_health_crisis": "mental_health_crisis",
        "public intox": "public_intoxication",
        "violent crime": "violent_crime",
        "assault/violent_crime": "violent_crime",
        "active_warrant": "warrant",
        "drunk_driving": "DUI",
        "suspicious person": "suspicious_person",
        "shoplifting": "retail_theft",
        "theft": "retail_theft"}))

In [ ]:
df_clean["suspected_offense_type"] = df_clean["suspected_offense_type"].replace({
    "dui": "DUI"
})

display(df_clean["suspected_offense_type"].value_counts(dropna=False))

In [ ]:
df_clean["suspected_offense_type"] = (
    df_clean["suspected_offense_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "mental health crisis": "mental_health_crisis",
        "behavioral_health_crisis": "mental_health_crisis",
        "public intox": "public_intoxication",
        "violent crime": "violent_crime",
        "assault/violent_crime": "violent_crime",
        "active_warrant": "warrant",
        "drunk_driving": "dui",
        "suspicious person": "suspicious_person",
        "shoplifting": "retail_theft",
        "theft": "retail_theft"
    })
)

In [ ]:
for col in ["suspect_age", "suspect_gender", "suspected_offense_type", "time_of_day"]:
    print(f"\n{col}:")
    print(df_clean[col].dropna().unique())

In [ ]:
print("Duplicate rows before removal:", df_clean.duplicated().sum())

In [ ]:
df_clean = df_clean.drop_duplicates()
print("Shape after dropping duplicates:", df_clean.shape)

In [ ]:
print("Cleaned dataset shape:", df_clean.shape)

print("\nMissing values:")
display(df_clean.isna().sum().sort_values(ascending=False))

print("\nCategorical value counts:")
display(df_clean["suspect_gender"].value_counts(dropna=False))
display(df_clean["suspected_offense_type"].value_counts(dropna=False))
display(df_clean["time_of_day"].value_counts(dropna=False))